# Coral Reef AI — End-to-End Validation

## Stage 3 YOLO11n → Predicted ROI + 10% Context → MobileNetV4 V3

This notebook evaluates the **integrated detection + classification pipeline** on the original **Validation split only**.

### Default models
- Detector: **YOLO11n Stage 3**
- Classifier: **MobileNetV4 Conv Small 0.5 — V3**
- Detector inference: **960 px**
- Detector confidence: **0.35** (selected previously from the Stage-3 validation audit)
- Matching IoU: **0.50**
- Classifier crop: **predicted detector bbox + 10% context**
- Classifier input: **288 × 288**

### Main end-to-end definition

A ground-truth coral counts as **End-to-End Correct** only when:

1. the detector predicts a box matched at `IoU >= 0.50`, **and**
2. the classifier predicts the correct operational coral label.

### Important
The detector **test split is not loaded or evaluated** in this notebook.

When Stage 4.1 is finished, only change `DETECTOR_CKPT` in Cell 2 and rerun the same notebook for a fair comparison.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + IMPORTS
# ============================================================

!pip -q install ultralytics==8.4.117 timm scikit-learn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict

import json
import math
import time
import random

import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import timm
from timm.data import resolve_model_data_config, create_transform

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from ultralytics import YOLO

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")

DEVICE = torch.device("cuda")
YOLO_DEVICE = 0

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("=" * 115)
print("ENVIRONMENT")
print("=" * 115)
print("GPU   :", GPU_NAME)
print(f"VRAM  : {VRAM_GB:.2f} GB")
print("Torch :", torch.__version__)
print("timm  :", timm.__version__)
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PATHS + MODEL SELECTION
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Coral_Reef_AI_Project"
)

MANIFEST = (
    PROJECT_ROOT
    / "04_Dataset_Splits"
    / "classification_roi_manifest_split.csv"
)

DET_IMAGES = (
    PROJECT_ROOT
    / "01_Data"
    / "Final"
    / "Detection_v1"
    / "images"
)

VAL_IMAGES = (
    DET_IMAGES
    / "validation"
)

# ------------------------------------------------------------
# CURRENT DETECTOR — STAGE 3
# ------------------------------------------------------------

STAGE3_BEST = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_TargetedSliced_SmallCrowded"
    / "Detection_Stage3"
    / "weights"
    / "best.pt"
)

DETECTOR_CKPT = STAGE3_BEST

# ------------------------------------------------------------
# WHEN STAGE 4.1 FINISHES:
# replace ONLY DETECTOR_CKPT with one selected checkpoint.
#
# Example:
#
# DETECTOR_CKPT = (
#     PROJECT_ROOT
#     / "06_Experiments"
#     / "YOLO11n_P2_Stage4_1_HybridRecovery"
#     / "Checkpoint_Audit"
#     / "best_stage41_map50_95.pt"
# )
# ------------------------------------------------------------


# ------------------------------------------------------------
# FINAL MOBILE CLASSIFIER — V3
# ------------------------------------------------------------

CLASSIFIER_CKPT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "MobileNetV4_Classification_V3_288_Context_HardMining"
    / "best_model_v3_macro_f1.pth"
)

FALLBACK_MODEL_NAME = (
    "mobilenetv4_conv_small_050.e3000_r224_in1k"
)

CLASSIFIER_INPUT = 288
CONTEXT_PAD = 0.10

# Established Stage-3 operating point.
DETECTOR_IMGSZ = 960
DETECTOR_CONF = 0.35
DETECTOR_NMS_IOU = 0.70
MATCH_IOU = 0.50

# Used only to label near-miss localization failures.
LOCALIZATION_NEAR_IOU = 0.10

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "EndToEnd_Stage3_MobileNetV4_V3_Validation"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

VIS_ROOT = OUTPUT_ROOT / "visuals"
VIS_ROOT.mkdir(parents=True, exist_ok=True)


for p in [
    MANIFEST,
    VAL_IMAGES,
    DETECTOR_CKPT,
    CLASSIFIER_CKPT,
]:
    if not p.exists():
        raise FileNotFoundError(p)


print("=" * 115)
print("END-TO-END CONFIGURATION")
print("=" * 115)

print("Detector:")
print(DETECTOR_CKPT)

print("\nClassifier:")
print(CLASSIFIER_CKPT)

print("\nDetector imgsz :", DETECTOR_IMGSZ)
print("Detector conf  :", DETECTOR_CONF)
print("Match IoU      :", MATCH_IOU)
print("Context pad    :", CONTEXT_PAD)
print("Classifier size:", CLASSIFIER_INPUT)

print("\nTEST SPLIT: NOT EVALUATED")
print("STATUS: PATHS_AND_CONFIG_READY")


In [ ]:
# ============================================================
# CELL 3 — LOAD VALIDATION GT FROM CLASSIFICATION MANIFEST
#
# We need the operational species labels, so the classification
# ROI manifest is the GT source for the integrated evaluation.
# ============================================================

df = pd.read_csv(MANIFEST)

required_columns = {
    "roi_id",
    "image_stem",
    "raw_detection_label",
    "x1_px",
    "y1_px",
    "x2_px",
    "y2_px",
    "split",
}

missing = required_columns - set(df.columns)

if missing:
    raise RuntimeError(
        f"Missing manifest columns: {sorted(missing)}"
    )

val_df = (
    df.loc[
        df["split"].astype(str) == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(val_df) == 354
assert val_df["image_stem"].nunique() == 100

CLASSES = sorted(
    val_df["raw_detection_label"]
    .astype(str)
    .unique()
    .tolist()
)

NUM_CLASSES = len(CLASSES)

assert NUM_CLASSES == 6

class_to_idx_manifest = {
    c: i
    for i, c in enumerate(CLASSES)
}

idx_to_class_manifest = {
    i: c
    for c, i in class_to_idx_manifest.items()
}

image_paths = []

for stem in sorted(
    val_df["image_stem"]
    .astype(str)
    .unique()
):
    p = VAL_IMAGES / f"{stem}.jpg"

    if not p.exists():
        raise FileNotFoundError(p)

    image_paths.append(p)

assert len(image_paths) == 100


gt_by_image = {}

for image_stem, g in val_df.groupby("image_stem"):
    rows = []

    for _, r in g.iterrows():
        rows.append({
            "roi_id": str(r["roi_id"]),
            "true_class": str(r["raw_detection_label"]),
            "bbox": np.array(
                [
                    float(r["x1_px"]),
                    float(r["y1_px"]),
                    float(r["x2_px"]),
                    float(r["y2_px"]),
                ],
                dtype=np.float32,
            ),
        })

    gt_by_image[str(image_stem)] = rows


print("=" * 115)
print("VALIDATION GROUND TRUTH")
print("=" * 115)

print("Images :", len(image_paths))
print("GT ROIs:", len(val_df))
print("Classes:", CLASSES)

print("\nPer-class GT:")
print(
    val_df["raw_detection_label"]
    .astype(str)
    .value_counts()
    .reindex(CLASSES)
)

print("\nTEST IMAGES WERE NOT LOADED.")
print("STATUS: VALIDATION_GT_READY")


In [ ]:
# ============================================================
# CELL 4 — LOAD YOLO STAGE3 + MOBILE V3
# ============================================================

# ------------------------------------------------------------
# Detector
# ------------------------------------------------------------

detector = YOLO(
    str(DETECTOR_CKPT)
)

print("=" * 115)
print("DETECTOR")
print("=" * 115)
print("Checkpoint:", DETECTOR_CKPT)

try:
    detector_strides = [
        int(round(float(x)))
        for x
        in detector.model.stride.detach().cpu().tolist()
    ]
    print("Strides:", detector_strides)
except Exception:
    detector_strides = None


# ------------------------------------------------------------
# Classifier checkpoint
# ------------------------------------------------------------

cls_ckpt = torch.load(
    CLASSIFIER_CKPT,
    map_location=DEVICE,
    weights_only=False,
)

MODEL_NAME = cls_ckpt.get(
    "model_name",
    FALLBACK_MODEL_NAME,
)

ckpt_class_to_idx = cls_ckpt.get(
    "class_to_idx",
    None,
)

if ckpt_class_to_idx is None:
    # V3 uses the same alphabetical ImageFolder ordering.
    ckpt_class_to_idx = dict(
        class_to_idx_manifest
    )

# Normalize mapping values in case checkpoint stores numpy ints.
ckpt_class_to_idx = {
    str(k): int(v)
    for k, v in ckpt_class_to_idx.items()
}

if set(ckpt_class_to_idx.keys()) != set(CLASSES):
    raise RuntimeError(
        "Classifier class names do not match validation manifest.\n"
        f"Checkpoint: {sorted(ckpt_class_to_idx.keys())}\n"
        f"Manifest:   {CLASSES}"
    )

idx_to_class = {
    int(v): str(k)
    for k, v in ckpt_class_to_idx.items()
}

classifier = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES,
).to(DEVICE)

classifier.load_state_dict(
    cls_ckpt["model_state_dict"]
)

classifier.eval()

checkpoint_input = int(
    cls_ckpt.get(
        "input_size",
        CLASSIFIER_INPUT,
    )
)

if checkpoint_input != CLASSIFIER_INPUT:
    print(
        f"WARNING: checkpoint input_size={checkpoint_input}; "
        f"notebook configured for {CLASSIFIER_INPUT}."
    )

data_cfg = resolve_model_data_config(
    classifier
)

data_cfg["input_size"] = (
    3,
    CLASSIFIER_INPUT,
    CLASSIFIER_INPUT,
)

if "mean" in cls_ckpt:
    data_cfg["mean"] = tuple(cls_ckpt["mean"])

if "std" in cls_ckpt:
    data_cfg["std"] = tuple(cls_ckpt["std"])

cls_transform = create_transform(
    **data_cfg,
    is_training=False,
)

print("\n" + "=" * 115)
print("CLASSIFIER")
print("=" * 115)

print("Model      :", MODEL_NAME)
print("Parameters :", f"{sum(p.numel() for p in classifier.parameters()) / 1e6:.3f} M")
print("Input      :", CLASSIFIER_INPUT)
print("Context    :", CONTEXT_PAD)
print("Classes    :", ckpt_class_to_idx)
print("V3 epoch   :", cls_ckpt.get("v3_epoch", cls_ckpt.get("epoch", "unknown")))
print("Stored F1  :", cls_ckpt.get("val_macro_f1", "unknown"))

print("\nSTATUS: MODELS_LOADED")


In [ ]:
# ============================================================
# CELL 5 — RUN STAGE3 DETECTOR ON ORIGINAL VALIDATION IMAGES
# ============================================================

print("=" * 115)
print("RUNNING DETECTOR")
print("=" * 115)

det_start = time.perf_counter()

det_results = detector.predict(
    source=[str(p) for p in image_paths],
    imgsz=DETECTOR_IMGSZ,

    # Low internal floor; we apply the fixed Stage3
    # operating threshold ourselves.
    conf=0.05,

    iou=DETECTOR_NMS_IOU,
    max_det=300,
    device=YOLO_DEVICE,
    verbose=False,
)

det_wall_sec = time.perf_counter() - det_start

assert len(det_results) == len(image_paths)

pred_by_image = {}
detector_inference_ms = []

for image_path, result in zip(image_paths, det_results):
    stem = image_path.stem

    if result.boxes is None or len(result.boxes) == 0:
        boxes_all = np.empty((0, 4), dtype=np.float32)
        conf_all = np.empty((0,), dtype=np.float32)

    else:
        boxes_all = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        conf_all = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    keep = conf_all >= DETECTOR_CONF

    pred_by_image[stem] = {
        "boxes": boxes_all[keep],
        "det_conf": conf_all[keep],
    }

    if (
        hasattr(result, "speed")
        and result.speed
        and "inference" in result.speed
    ):
        detector_inference_ms.append(
            float(result.speed["inference"])
        )


total_predictions = sum(
    len(v["boxes"])
    for v in pred_by_image.values()
)

mean_detector_model_ms = (
    float(np.mean(detector_inference_ms))
    if detector_inference_ms
    else None
)

detector_wall_ms_per_image = (
    det_wall_sec / len(image_paths) * 1000.0
)

preds_per_frame = np.array(
    [
        len(pred_by_image[p.stem]["boxes"])
        for p in image_paths
    ],
    dtype=np.int64,
)

print("Validation images        :", len(image_paths))
print("Predictions @ conf>=.35  :", total_predictions)
print("Mean predictions / frame :", f"{preds_per_frame.mean():.2f}")
print("Median predictions/frame :", f"{np.median(preds_per_frame):.1f}")
print("Max predictions / frame  :", int(preds_per_frame.max()))

if mean_detector_model_ms is not None:
    print(
        "YOLO reported inference  :",
        f"{mean_detector_model_ms:.2f} ms/image"
    )

print(
    "Detector wall throughput :",
    f"{detector_wall_ms_per_image:.2f} ms/image"
)

print("STATUS: DETECTION_COMPLETE")


In [ ]:
# ============================================================
# CELL 6 — CROP PREDICTED BOXES +10% AND CLASSIFY WITH V3
# ============================================================

def padded_crop_from_xyxy(
    pil_image,
    box,
    padding=0.10,
):
    W, H = pil_image.size

    x1, y1, x2, y2 = map(float, box)

    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)

    if bw <= 0 or bh <= 0:
        return None, None

    pad_x = bw * padding
    pad_y = bh * padding

    px1 = max(
        0,
        int(math.floor(x1 - pad_x))
    )

    py1 = max(
        0,
        int(math.floor(y1 - pad_y))
    )

    px2 = min(
        W,
        int(math.ceil(x2 + pad_x))
    )

    py2 = min(
        H,
        int(math.ceil(y2 + pad_y))
    )

    if px2 <= px1 or py2 <= py1:
        return None, None

    crop = pil_image.crop(
        (px1, py1, px2, py2)
    )

    return crop, (px1, py1, px2, py2)


crop_tensors = []
crop_meta = []

crop_build_start = time.perf_counter()

for image_path in image_paths:
    stem = image_path.stem

    with Image.open(image_path) as im:
        image_rgb = im.convert("RGB")

    boxes = pred_by_image[stem]["boxes"]
    det_confs = pred_by_image[stem]["det_conf"]

    for pred_idx, (box, det_conf) in enumerate(
        zip(boxes, det_confs)
    ):
        crop, padded_box = padded_crop_from_xyxy(
            image_rgb,
            box,
            padding=CONTEXT_PAD,
        )

        if crop is None:
            continue

        tensor = cls_transform(
            crop
        )

        crop_tensors.append(
            tensor
        )

        crop_meta.append({
            "image_stem": stem,
            "pred_index": int(pred_idx),
            "det_box": np.asarray(box, dtype=np.float32),
            "padded_crop_box": padded_box,
            "det_conf": float(det_conf),
        })

crop_build_sec = time.perf_counter() - crop_build_start

assert len(crop_tensors) == total_predictions, (
    f"Expected {total_predictions} valid crops, "
    f"built {len(crop_tensors)}."
)


CLASSIFIER_BATCH = 64 if VRAM_GB >= 12 else 32

all_pred_indices = []
all_cls_conf = []
all_probs = []

cls_start = time.perf_counter()

with torch.no_grad():
    for start_idx in range(
        0,
        len(crop_tensors),
        CLASSIFIER_BATCH,
    ):
        batch = torch.stack(
            crop_tensors[
                start_idx:start_idx + CLASSIFIER_BATCH
            ]
        ).to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = classifier(batch)

        probs = torch.softmax(
            logits.float(),
            dim=1,
        )

        confs, preds = probs.max(dim=1)

        all_pred_indices.extend(
            preds.cpu().tolist()
        )

        all_cls_conf.extend(
            confs.cpu().tolist()
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

cls_sec = time.perf_counter() - cls_start

all_probs = np.asarray(
    all_probs,
    dtype=np.float32,
)

for i, meta in enumerate(crop_meta):
    cls_idx = int(all_pred_indices[i])

    meta["predicted_class_index"] = cls_idx
    meta["predicted_class"] = idx_to_class[cls_idx]
    meta["classifier_conf"] = float(all_cls_conf[i])

    # Useful ranking score only; not calibrated probability.
    meta["combined_conf"] = float(
        meta["det_conf"]
        * meta["classifier_conf"]
    )

    meta["class_probs"] = all_probs[i]


classified_by_image = defaultdict(list)

for meta in crop_meta:
    classified_by_image[
        meta["image_stem"]
    ].append(meta)


classifier_ms_per_crop = (
    cls_sec / max(len(crop_meta), 1) * 1000.0
)

crop_build_ms_per_crop = (
    crop_build_sec / max(len(crop_meta), 1) * 1000.0
)

classifier_ms_per_image = (
    cls_sec / len(image_paths) * 1000.0
)

print("=" * 115)
print("PREDICTED-ROI CLASSIFICATION")
print("=" * 115)

print("Predicted crops        :", len(crop_meta))
print("Classifier batch       :", CLASSIFIER_BATCH)
print("Crop build             :", f"{crop_build_ms_per_crop:.3f} ms/crop")
print("Classifier throughput  :", f"{classifier_ms_per_crop:.3f} ms/crop")
print("Classifier batch wall  :", f"{classifier_ms_per_image:.2f} ms/image")

print("STATUS: CLASSIFICATION_ON_PREDICTED_ROIS_COMPLETE")


In [ ]:
# ============================================================
# CELL 7 — GREEDY DETECTION MATCHING + END-TO-END ERROR TABLES
# ============================================================

def iou_matrix(a, b):
    a = np.asarray(a, dtype=np.float32).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float32).reshape(-1, 4)

    if len(a) == 0 or len(b) == 0:
        return np.zeros(
            (len(a), len(b)),
            dtype=np.float32,
        )

    lt = np.maximum(
        a[:, None, :2],
        b[None, :, :2],
    )

    rb = np.minimum(
        a[:, None, 2:],
        b[None, :, 2:],
    )

    wh = np.clip(
        rb - lt,
        0,
        None,
    )

    inter = wh[..., 0] * wh[..., 1]

    area_a = (
        np.clip(a[:, 2] - a[:, 0], 0, None)
        * np.clip(a[:, 3] - a[:, 1], 0, None)
    )

    area_b = (
        np.clip(b[:, 2] - b[:, 0], 0, None)
        * np.clip(b[:, 3] - b[:, 1], 0, None)
    )

    union = (
        area_a[:, None]
        + area_b[None, :]
        - inter
    )

    return inter / np.clip(
        union,
        1e-9,
        None,
    )


def greedy_match(
    pred_boxes,
    gt_boxes,
    threshold=0.50,
):
    pred_boxes = np.asarray(
        pred_boxes,
        dtype=np.float32,
    ).reshape(-1, 4)

    gt_boxes = np.asarray(
        gt_boxes,
        dtype=np.float32,
    ).reshape(-1, 4)

    if len(pred_boxes) == 0:
        return [], [], list(range(len(gt_boxes)))

    if len(gt_boxes) == 0:
        return [], list(range(len(pred_boxes))), []

    ious = iou_matrix(
        pred_boxes,
        gt_boxes,
    )

    candidates = []

    for p in range(len(pred_boxes)):
        for g in range(len(gt_boxes)):
            if ious[p, g] >= threshold:
                candidates.append(
                    (float(ious[p, g]), p, g)
                )

    candidates.sort(reverse=True)

    used_p = set()
    used_g = set()
    matches = []

    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue

        used_p.add(p)
        used_g.add(g)

        matches.append(
            (p, g, iou)
        )

    unmatched_p = [
        p
        for p in range(len(pred_boxes))
        if p not in used_p
    ]

    unmatched_g = [
        g
        for g in range(len(gt_boxes))
        if g not in used_g
    ]

    return (
        matches,
        unmatched_p,
        unmatched_g,
    )


gt_rows = []
prediction_rows = []
image_rows = []

for image_path in image_paths:
    stem = image_path.stem

    gt_items = gt_by_image[stem]
    pred_items = classified_by_image.get(
        stem,
        [],
    )

    gt_boxes = np.asarray(
        [x["bbox"] for x in gt_items],
        dtype=np.float32,
    ).reshape(-1, 4)

    pred_boxes = np.asarray(
        [x["det_box"] for x in pred_items],
        dtype=np.float32,
    ).reshape(-1, 4)

    matches, unmatched_pred, unmatched_gt = greedy_match(
        pred_boxes,
        gt_boxes,
        MATCH_IOU,
    )

    match_by_gt = {
        g: (p, iou)
        for p, g, iou in matches
    }

    match_by_pred = {
        p: (g, iou)
        for p, g, iou in matches
    }

    all_ious = iou_matrix(
        pred_boxes,
        gt_boxes,
    )

    image_correct = 0
    image_cls_errors = 0

    # --------------------------------------------------------
    # GT-centric rows
    # --------------------------------------------------------

    for g_idx, gt in enumerate(gt_items):
        true_class = gt["true_class"]

        if g_idx in match_by_gt:
            p_idx, iou = match_by_gt[g_idx]
            pred = pred_items[p_idx]

            pred_class = pred["predicted_class"]
            class_correct = (
                pred_class == true_class
            )

            failure_mode = (
                "correct"
                if class_correct
                else "classification_error"
            )

            if class_correct:
                image_correct += 1
            else:
                image_cls_errors += 1

            gt_rows.append({
                "image_stem": stem,
                "roi_id": gt["roi_id"],
                "true_class": true_class,

                "gt_x1": float(gt["bbox"][0]),
                "gt_y1": float(gt["bbox"][1]),
                "gt_x2": float(gt["bbox"][2]),
                "gt_y2": float(gt["bbox"][3]),

                "detected": True,
                "matched_pred_index": int(p_idx),
                "iou": float(iou),

                "det_conf": float(pred["det_conf"]),
                "predicted_class": pred_class,
                "classifier_conf": float(pred["classifier_conf"]),
                "combined_conf": float(pred["combined_conf"]),

                "class_correct": bool(class_correct),
                "end_to_end_correct": bool(class_correct),
                "failure_mode": failure_mode,
            })

        else:
            if len(pred_boxes):
                best_possible_iou = float(
                    all_ious[:, g_idx].max()
                )
            else:
                best_possible_iou = 0.0

            if (
                best_possible_iou
                >= LOCALIZATION_NEAR_IOU
                and best_possible_iou
                < MATCH_IOU
            ):
                failure_mode = "localization_error"
            else:
                failure_mode = "detector_miss"

            gt_rows.append({
                "image_stem": stem,
                "roi_id": gt["roi_id"],
                "true_class": true_class,

                "gt_x1": float(gt["bbox"][0]),
                "gt_y1": float(gt["bbox"][1]),
                "gt_x2": float(gt["bbox"][2]),
                "gt_y2": float(gt["bbox"][3]),

                "detected": False,
                "matched_pred_index": None,
                "iou": float(best_possible_iou),

                "det_conf": np.nan,
                "predicted_class": None,
                "classifier_conf": np.nan,
                "combined_conf": np.nan,

                "class_correct": False,
                "end_to_end_correct": False,
                "failure_mode": failure_mode,
            })

    # --------------------------------------------------------
    # Prediction-centric rows
    # --------------------------------------------------------

    for p_idx, pred in enumerate(pred_items):
        if p_idx in match_by_pred:
            g_idx, iou = match_by_pred[p_idx]
            gt = gt_items[g_idx]

            matched = True
            true_class = gt["true_class"]
            roi_id = gt["roi_id"]

            class_correct = (
                pred["predicted_class"]
                == true_class
            )

            pred_status = (
                "correct"
                if class_correct
                else "wrong_species"
            )

        else:
            matched = False
            true_class = None
            roi_id = None
            class_correct = False

            if len(gt_boxes):
                max_iou = float(
                    all_ious[p_idx, :].max()
                )
            else:
                max_iou = 0.0

            pred_status = (
                "localization_fp"
                if (
                    max_iou >= LOCALIZATION_NEAR_IOU
                    and max_iou < MATCH_IOU
                )
                else "pure_fp"
            )

            iou = max_iou

        box = pred["det_box"]

        prediction_rows.append({
            "image_stem": stem,
            "pred_index": int(p_idx),

            "pred_x1": float(box[0]),
            "pred_y1": float(box[1]),
            "pred_x2": float(box[2]),
            "pred_y2": float(box[3]),

            "det_conf": float(pred["det_conf"]),

            "predicted_class": pred["predicted_class"],
            "classifier_conf": float(pred["classifier_conf"]),
            "combined_conf": float(pred["combined_conf"]),

            "matched": bool(matched),
            "matched_roi_id": roi_id,
            "true_class": true_class,
            "iou": float(iou),

            "class_correct": bool(class_correct),
            "end_to_end_correct": bool(
                matched and class_correct
            ),

            "prediction_status": pred_status,
        })

    image_rows.append({
        "image_stem": stem,
        "gt_count": len(gt_items),
        "prediction_count": len(pred_items),
        "matched_detections": len(matches),
        "correct_species": image_correct,
        "classification_errors": image_cls_errors,
        "detector_fn": len(unmatched_gt),
        "detector_fp": len(unmatched_pred),
        "total_e2e_errors": (
            image_cls_errors
            + len(unmatched_gt)
            + len(unmatched_pred)
        ),
    })


gt_eval_df = pd.DataFrame(gt_rows)
pred_eval_df = pd.DataFrame(prediction_rows)
image_eval_df = pd.DataFrame(image_rows)

assert len(gt_eval_df) == 354
assert len(pred_eval_df) == total_predictions


GT_EVAL_CSV = (
    OUTPUT_ROOT
    / "end_to_end_gt_outcomes.csv"
)

PRED_EVAL_CSV = (
    OUTPUT_ROOT
    / "end_to_end_prediction_outcomes.csv"
)

IMAGE_EVAL_CSV = (
    OUTPUT_ROOT
    / "end_to_end_per_image_summary.csv"
)

gt_eval_df.to_csv(
    GT_EVAL_CSV,
    index=False,
)

pred_eval_df.to_csv(
    PRED_EVAL_CSV,
    index=False,
)

image_eval_df.to_csv(
    IMAGE_EVAL_CSV,
    index=False,
)

print("=" * 115)
print("MATCHING + ERROR TABLES")
print("=" * 115)

print(
    gt_eval_df["failure_mode"]
    .value_counts()
)

print("\nPrediction outcomes:")
print(
    pred_eval_df["prediction_status"]
    .value_counts()
)

print("\nSTATUS: END_TO_END_TABLES_READY")


In [ ]:
# ============================================================
# CELL 8 — END-TO-END METRICS + PER-CLASS ANALYSIS
# ============================================================

TOTAL_GT = len(gt_eval_df)
TOTAL_PRED = len(pred_eval_df)

MATCHED = int(
    gt_eval_df["detected"].sum()
)

CORRECT_E2E = int(
    gt_eval_df["end_to_end_correct"].sum()
)

CLASS_ERRORS = int(
    (
        gt_eval_df["failure_mode"]
        == "classification_error"
    ).sum()
)

LOCALIZATION_ERRORS = int(
    (
        gt_eval_df["failure_mode"]
        == "localization_error"
    ).sum()
)

PURE_DETECTOR_MISSES = int(
    (
        gt_eval_df["failure_mode"]
        == "detector_miss"
    ).sum()
)

UNMATCHED_PRED = int(
    (~pred_eval_df["matched"]).sum()
)


# ------------------------------------------------------------
# Detection metrics at the fixed Stage3 operating threshold.
# ------------------------------------------------------------

det_precision = (
    MATCHED / TOTAL_PRED
    if TOTAL_PRED
    else 0.0
)

det_recall = (
    MATCHED / TOTAL_GT
    if TOTAL_GT
    else 0.0
)

det_f1 = (
    2 * det_precision * det_recall
    / (det_precision + det_recall)
    if (det_precision + det_recall)
    else 0.0
)


# ------------------------------------------------------------
# Classifier accuracy ONLY on correctly localized matched boxes.
# ------------------------------------------------------------

cls_acc_on_matched = (
    CORRECT_E2E / MATCHED
    if MATCHED
    else 0.0
)


# ------------------------------------------------------------
# End-to-end species metrics.
#
# A final output is correct only if:
# detection matched + species correct.
#
# Any unmatched prediction or matched wrong-species prediction
# is not a correct final species output.
# ------------------------------------------------------------

e2e_precision = (
    CORRECT_E2E / TOTAL_PRED
    if TOTAL_PRED
    else 0.0
)

e2e_recall = (
    CORRECT_E2E / TOTAL_GT
    if TOTAL_GT
    else 0.0
)

e2e_f1 = (
    2 * e2e_precision * e2e_recall
    / (e2e_precision + e2e_recall)
    if (e2e_precision + e2e_recall)
    else 0.0
)


# ------------------------------------------------------------
# Matched-detection classifier report.
# ------------------------------------------------------------

matched_df = gt_eval_df[
    gt_eval_df["detected"]
].copy()

y_true_matched = matched_df[
    "true_class"
].tolist()

y_pred_matched = matched_df[
    "predicted_class"
].tolist()

matched_classification_report = classification_report(
    y_true_matched,
    y_pred_matched,
    labels=CLASSES,
    zero_division=0,
)

matched_macro_f1 = f1_score(
    y_true_matched,
    y_pred_matched,
    labels=CLASSES,
    average="macro",
    zero_division=0,
)


# ------------------------------------------------------------
# Per-class end-to-end metrics.
# ------------------------------------------------------------

per_class_rows = []

for cls in CLASSES:
    g = gt_eval_df[
        gt_eval_df["true_class"] == cls
    ]

    gt_count = len(g)
    detected_count = int(
        g["detected"].sum()
    )
    correct_count = int(
        g["end_to_end_correct"].sum()
    )

    det_r = (
        detected_count / gt_count
        if gt_count
        else 0.0
    )

    cls_acc_detected = (
        correct_count / detected_count
        if detected_count
        else 0.0
    )

    e2e_r = (
        correct_count / gt_count
        if gt_count
        else 0.0
    )

    predicted_as_cls = int(
        (
            pred_eval_df["predicted_class"]
            == cls
        ).sum()
    )

    e2e_p_cls = (
        correct_count / predicted_as_cls
        if predicted_as_cls
        else 0.0
    )

    e2e_f1_cls = (
        2 * e2e_p_cls * e2e_r
        / (e2e_p_cls + e2e_r)
        if (e2e_p_cls + e2e_r)
        else 0.0
    )

    per_class_rows.append({
        "class": cls,
        "GT": gt_count,
        "detected": detected_count,
        "correct_species": correct_count,

        "detector_recall": det_r,
        "classifier_accuracy_on_detected": cls_acc_detected,

        "end_to_end_precision": e2e_p_cls,
        "end_to_end_recall": e2e_r,
        "end_to_end_f1": e2e_f1_cls,
    })


per_class_df = pd.DataFrame(
    per_class_rows
)

PER_CLASS_CSV = (
    OUTPUT_ROOT
    / "end_to_end_per_class_metrics.csv"
)

per_class_df.to_csv(
    PER_CLASS_CSV,
    index=False,
)


print("=" * 125)
print("END-TO-END VALIDATION RESULTS")
print("=" * 125)

print(f"GT coral instances                   : {TOTAL_GT}")
print(f"Final detector predictions           : {TOTAL_PRED}")
print(f"Matched detections IoU >= {MATCH_IOU:.2f}       : {MATCHED}")
print(f"Correct detection + correct species  : {CORRECT_E2E}")

print("\n--- DETECTOR AT FIXED OPERATING POINT ---")
print(f"Precision                            : {det_precision:.4f}")
print(f"Recall                               : {det_recall:.4f}")
print(f"F1                                   : {det_f1:.4f}")

print("\n--- CLASSIFIER ON MATCHED DETECTIONS ---")
print(f"Accuracy                             : {cls_acc_on_matched:.4f}")
print(f"Macro-F1                             : {matched_macro_f1:.4f}")

print("\n--- TRUE END-TO-END SPECIES PERFORMANCE ---")
print(f"End-to-End Species Precision         : {e2e_precision:.4f}")
print(f"End-to-End Species Recall            : {e2e_recall:.4f}")
print(f"End-to-End Species F1                : {e2e_f1:.4f}")

print("\n--- FAILURE DECOMPOSITION ---")
print(f"Classification errors                : {CLASS_ERRORS}")
print(f"Localization errors / near misses    : {LOCALIZATION_ERRORS}")
print(f"Pure detector misses                 : {PURE_DETECTOR_MISSES}")
print(f"Unmatched detector predictions       : {UNMATCHED_PRED}")

print("\n" + "=" * 125)
print("CLASSIFIER REPORT — MATCHED DETECTIONS ONLY")
print("=" * 125)
print(matched_classification_report)

print("\n" + "=" * 125)
print("PER-CLASS END-TO-END METRICS")
print("=" * 125)

print(
    per_class_df.to_string(
        index=False,
        formatters={
            "detector_recall": "{:.4f}".format,
            "classifier_accuracy_on_detected": "{:.4f}".format,
            "end_to_end_precision": "{:.4f}".format,
            "end_to_end_recall": "{:.4f}".format,
            "end_to_end_f1": "{:.4f}".format,
        },
    )
)

print("\nSTATUS: END_TO_END_METRICS_COMPLETE")


In [ ]:
# ============================================================
# CELL 9 — CONFUSION MATRIX + LATENCY + FINAL SUMMARY
# ============================================================

# ------------------------------------------------------------
# Confusion matrix for classifier conditional on successful
# detector localization.
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true_matched,
    y_pred_matched,
    labels=CLASSES,
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

ax.imshow(cm)

ax.set_xticks(
    range(NUM_CLASSES)
)

ax.set_yticks(
    range(NUM_CLASSES)
)

ax.set_xticklabels(
    CLASSES,
    rotation=45,
    ha="right",
)

ax.set_yticklabels(
    CLASSES
)

ax.set_xlabel(
    "Predicted Species"
)

ax.set_ylabel(
    "True Species"
)

ax.set_title(
    "End-to-End Validation — Classifier Confusion on Matched Detections"
)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
        )

fig.tight_layout()

CM_PATH = (
    OUTPUT_ROOT
    / "matched_detection_species_confusion_matrix.png"
)

fig.savefig(
    CM_PATH,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Latency / throughput estimate
# ------------------------------------------------------------

pipeline_wall_sec = (
    det_wall_sec
    + crop_build_sec
    + cls_sec
)

pipeline_ms_per_image = (
    pipeline_wall_sec
    / len(image_paths)
    * 1000.0
)

pipeline_fps = (
    1000.0 / pipeline_ms_per_image
    if pipeline_ms_per_image > 0
    else 0.0
)


print("=" * 115)
print("PIPELINE THROUGHPUT")
print("=" * 115)

if mean_detector_model_ms is not None:
    print(
        "YOLO reported inference      :",
        f"{mean_detector_model_ms:.2f} ms/image"
    )

print(
    "Detector batch wall           :",
    f"{detector_wall_ms_per_image:.2f} ms/image"
)

print(
    "Crop preparation              :",
    f"{crop_build_ms_per_crop:.3f} ms/crop"
)

print(
    "Classifier throughput         :",
    f"{classifier_ms_per_crop:.3f} ms/crop"
)

print(
    "Predictions / frame mean      :",
    f"{preds_per_frame.mean():.2f}"
)

print(
    "Integrated wall time          :",
    f"{pipeline_ms_per_image:.2f} ms/image"
)

print(
    "Integrated throughput         :",
    f"{pipeline_fps:.2f} FPS"
)

print(
    "\nNOTE: classifier is batched across validation crops, "
    "so this is throughput-oriented rather than strict "
    "single-frame real-time latency."
)


# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

summary = {
    "experiment": (
        "Stage3_YOLO11n_to_MobileNetV4_V3_EndToEnd_Validation"
    ),

    "detector_checkpoint": str(
        DETECTOR_CKPT
    ),

    "classifier_checkpoint": str(
        CLASSIFIER_CKPT
    ),

    "validation_images": int(
        len(image_paths)
    ),

    "validation_gt_instances": int(
        TOTAL_GT
    ),

    "detector_imgsz": int(
        DETECTOR_IMGSZ
    ),

    "detector_confidence": float(
        DETECTOR_CONF
    ),

    "matching_iou": float(
        MATCH_IOU
    ),

    "classifier_input_size": int(
        CLASSIFIER_INPUT
    ),

    "classifier_context_padding": float(
        CONTEXT_PAD
    ),

    "detector_precision": float(
        det_precision
    ),

    "detector_recall": float(
        det_recall
    ),

    "detector_f1": float(
        det_f1
    ),

    "matched_detection_classifier_accuracy": float(
        cls_acc_on_matched
    ),

    "matched_detection_classifier_macro_f1": float(
        matched_macro_f1
    ),

    "end_to_end_species_precision": float(
        e2e_precision
    ),

    "end_to_end_species_recall": float(
        e2e_recall
    ),

    "end_to_end_species_f1": float(
        e2e_f1
    ),

    "correct_end_to_end": int(
        CORRECT_E2E
    ),

    "classification_errors": int(
        CLASS_ERRORS
    ),

    "localization_errors": int(
        LOCALIZATION_ERRORS
    ),

    "pure_detector_misses": int(
        PURE_DETECTOR_MISSES
    ),

    "unmatched_predictions": int(
        UNMATCHED_PRED
    ),

    "mean_predictions_per_frame": float(
        preds_per_frame.mean()
    ),

    "mean_detector_reported_inference_ms": (
        float(mean_detector_model_ms)
        if mean_detector_model_ms is not None
        else None
    ),

    "integrated_wall_ms_per_image": float(
        pipeline_ms_per_image
    ),

    "integrated_throughput_fps": float(
        pipeline_fps
    ),

    "test_evaluated": False,
}


SUMMARY_JSON = (
    OUTPUT_ROOT
    / "end_to_end_validation_summary.json"
)

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("\nSaved:")
print(SUMMARY_JSON)
print(GT_EVAL_CSV)
print(PRED_EVAL_CSV)
print(IMAGE_EVAL_CSV)
print(PER_CLASS_CSV)
print(CM_PATH)

print("\nTEST SET WAS NOT EVALUATED.")
print("STATUS: END_TO_END_VALIDATION_COMPLETE")


In [ ]:
# ============================================================
# CELL 10 — SAVE HARDEST VALIDATION VISUALS
#
# Predicted box colors:
#   Green  = matched + correct species
#   Orange = matched + wrong species
#   Red    = unmatched prediction
#
# GT misses are drawn in blue.
# ============================================================

TOP_N_VISUALS = 12

hardest_images = (
    image_eval_df
    .sort_values(
        by=[
            "total_e2e_errors",
            "detector_fn",
            "classification_errors",
            "detector_fp",
        ],
        ascending=False,
    )
    .head(TOP_N_VISUALS)
    ["image_stem"]
    .tolist()
)


def draw_text_with_bg(
    draw,
    xy,
    text,
    fill,
):
    x, y = xy

    bbox = draw.textbbox(
        (x, y),
        text,
    )

    pad = 2

    draw.rectangle(
        (
            bbox[0] - pad,
            bbox[1] - pad,
            bbox[2] + pad,
            bbox[3] + pad,
        ),
        fill=fill,
    )

    draw.text(
        (x, y),
        text,
        fill=(255, 255, 255),
    )


for stem in hardest_images:
    image_path = VAL_IMAGES / f"{stem}.jpg"

    with Image.open(image_path) as im:
        vis = im.convert("RGB")

    draw = ImageDraw.Draw(vis)

    p_rows = pred_eval_df[
        pred_eval_df["image_stem"] == stem
    ]

    g_rows = gt_eval_df[
        gt_eval_df["image_stem"] == stem
    ]

    # --------------------------------------------------------
    # Draw unmatched GT misses first.
    # --------------------------------------------------------

    for _, r in g_rows.iterrows():
        if bool(r["detected"]):
            continue

        box = (
            float(r["gt_x1"]),
            float(r["gt_y1"]),
            float(r["gt_x2"]),
            float(r["gt_y2"]),
        )

        draw.rectangle(
            box,
            outline=(0, 100, 255),
            width=4,
        )

        label = (
            f"MISS {r['true_class']} "
            f"[{r['failure_mode']}]"
        )

        draw_text_with_bg(
            draw,
            (box[0], max(0, box[1] - 16)),
            label,
            (0, 80, 200),
        )

    # --------------------------------------------------------
    # Draw predictions.
    # --------------------------------------------------------

    for _, r in p_rows.iterrows():
        box = (
            float(r["pred_x1"]),
            float(r["pred_y1"]),
            float(r["pred_x2"]),
            float(r["pred_y2"]),
        )

        status = r["prediction_status"]

        if status == "correct":
            color = (0, 180, 0)

        elif status == "wrong_species":
            color = (255, 140, 0)

        else:
            color = (220, 0, 0)

        draw.rectangle(
            box,
            outline=color,
            width=4,
        )

        if bool(r["matched"]):
            label = (
                f"{r['predicted_class']} "
                f"D:{r['det_conf']:.2f} "
                f"C:{r['classifier_conf']:.2f} "
                f"IoU:{r['iou']:.2f}"
            )
        else:
            label = (
                f"FP {r['predicted_class']} "
                f"D:{r['det_conf']:.2f} "
                f"C:{r['classifier_conf']:.2f}"
            )

        draw_text_with_bg(
            draw,
            (box[0], max(0, box[1] - 16)),
            label,
            color,
        )

    out_path = (
        VIS_ROOT
        / f"{stem}_e2e.jpg"
    )

    vis.save(
        out_path,
        "JPEG",
        quality=92,
    )


print("=" * 115)
print("HARDEST VALIDATION VISUALS")
print("=" * 115)

print("Saved:", len(hardest_images))
print("Folder:", VIS_ROOT)

print("\nImages:")
for x in hardest_images:
    print(" ", x)

print("\nSTATUS: ERROR_VISUALS_SAVED")


## How to compare Stage 4.1 later

Do **not** change the classifier, validation data, thresholds, matching logic, or context padding.

Change only this line in **Cell 2**:

```python
DETECTOR_CKPT = STAGE3_BEST
```

to the selected Stage-4.1 checkpoint, then rerun the notebook.

The fair comparison should use:

- `End-to-End Species Recall`
- `End-to-End Species Precision`
- `End-to-End Species F1`
- classifier accuracy on matched detections
- per-class End-to-End Recall
- detector misses vs classifier errors
- integrated throughput
